# 02 - Embeddings, vector search & friends 🧭

Retrieval is the **R** in RAG. If the right film isn't retrieved, the LLM can't recommend it.

We'll build four retrievers and compare them:

1. **Vector search**: meaning-based (embeddings)
2. **BM25**: keyword-based (classic search engines)
3. **Hybrid**: both, merged with Reciprocal Rank Fusion
4. **+ Reranker**: a slower, sharper model re-orders the top results

In [1]:
# Make the project importable even if it wasn't installed (uv sync installs it)
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

%load_ext autoreload
%autoreload 2

from movierag.config import get_config
cfg = get_config()
print(cfg.summary())

bookmarks     : /home/jp/llms/rag/movies-reels.json
state dir     : /home/jp/.movierag
LLM           : hf:Qwen/Qwen3-4B-Instruct-2507 (4-bit)
embeddings    : BAAI/bge-small-en-v1.5
reranker      : cross-encoder/ms-marco-MiniLM-L6-v2
TMDb key      : yes
YouTube key   : MISSING
watch region  : CH
offline mode  : False


In [2]:
from movierag.catalogue import load_catalogue
titles = load_catalogue(cfg.catalogue_path)
print(len(titles), "titles")

85 titles


## 1. What is an embedding? 🔢

In notebook 02 of the workshop, `nn.Embedding` gave each **token** a learned vector.

A **sentence embedding model** does the same for a whole text: it reads the text with a
small Transformer and pools the token vectors into one vector.

Texts with similar meaning → vectors pointing in similar directions.

We use `BAAI/bge-small-en-v1.5`: 33M parameters, 384 numbers per text, runs fine on CPU.

In [3]:
from movierag.models import load_embed_model, gpu_report
embed_model = load_embed_model(cfg)
print(gpu_report())

v = embed_model.get_text_embedding("A submarine captain defies the US Navy")
print(len(v), "numbers:", [round(x, 3) for x in v[:8]], "...")

/home/jp/llms/rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1490.66it/s]


NVIDIA RTX PRO 2000 Blackwell Generation Laptop GPU: 1.2 GB used / 8.0 GB
384 numbers: [-0.046, 0.047, 0.042, -0.035, 0.027, 0.007, 0.082, 0.08] ...


### Similar meaning, similar direction 📐

**Cosine similarity** = how aligned two vectors are (1 = same direction, 0 = unrelated).

In [4]:
import numpy as np

sentences = [
    "A submarine captain defies the US Navy",
    "Tense standoff between a nuclear sub and an aircraft carrier",
    "A teenage girl learns to bake croissants in Paris",
]
vecs = np.array([embed_model.get_text_embedding(s) for s in sentences])
sims = vecs @ vecs.T   # vectors are normalised, so dot product = cosine similarity

for i, s in enumerate(sentences):
    print(f"{s[:45]:47}", " ".join(f"{x:5.2f}" for x in sims[i]))

A submarine captain defies the US Navy           1.00  0.64  0.29
Tense standoff between a nuclear sub and an a    0.64  1.00  0.28
A teenage girl learns to bake croissants in P    0.29  0.28  1.00


💡 Sentences 1 and 2 share almost **no words**, but their vectors are close.
That is what keyword search can't do.

## 2. Build the index 🗂️

One **node** per title (our texts are short, so no chunking needed).

Vectors are stored in **Chroma**, a small vector database that persists to disk
(in `MOVIERAG_HOME`, outside Google Drive).

LlamaIndex vocabulary:

- `TextNode`: a piece of text + metadata + an ID
- `VectorStoreIndex`: embeds nodes and stores them in a vector store
- `Retriever`: query → list of `NodeWithScore`

In [8]:
from movierag.index import MovieIndex

index = MovieIndex(titles, embed_model, persist_dir=cfg.chroma_dir)
print(index.nodes[0].text)
print(index.nodes[0].metadata)

The Looking Planet (2014) - short
Genres: Adventure, Animation
Plot: During the construction of the universe, a young member of the Cosmos Corps of Engineers decides to break some fundamental laws in the name of self-expression.
Keywords: short film
By: Eric Law Anderson
Cast: Samuel Hery, Joe Cappelletti
{'item_id': 'movie:375729', 'title': 'The Looking Planet', 'kind': 'short', 'source': 'my_list', 'year': 2014, 'runtime_min': 16, 'genres': 'Adventure|Animation'}


In [9]:
def show(hits, n=5):
    for t, score in index.titles_for(hits)[:n]:
        print(f"  {score:7.3f}  {t.title} ({t.year}) - {t.kind}")

## 3. Vector search vs BM25 🥊

**BM25** scores documents by the query words they contain, weighted by how rare
each word is (rare words like *Paganini* count a lot, common ones like *the* barely count).

Try both on queries that favour one or the other.

In [10]:
queries = [
    "Paganini",                                          # rare name: BM25 territory
    "a man stranded on a lifeboat with a wild animal",   # paraphrase: vector territory
    "Shaolin monks",
    "eerie arthouse film about an alien seductress",
]
for q in queries:
    print(f"\n=== {q}")
    print(" vector:"); show(index.vector_search(q, top_k=3), 3)
    print(" bm25:");   show(index.bm25_search(q, top_k=3), 3)


=== Paganini
 vector:
    0.743  The Devil's Violinist (2013) - film
    0.633  Black Narcissus (1947) - film
    0.625  The Last Witch Hunter (2015) - film
 bm25:
    2.807  The Devil's Violinist (2013) - film

=== a man stranded on a lifeboat with a wild animal
 vector:
    0.614  Life of Pi (2012) - film
    0.605  The Abyss (1989) - film
    0.601  Mickey 17 (2025) - film
 bm25:
    2.630  Life of Pi (2012) - film
    2.398  Masters of the Universe (1987) - film
    2.069  Memories (1995) - film

=== Shaolin monks
 vector:
    0.704  The 18 Bronzemen (1976) - film
    0.698  Monk Comes Down the Mountain (2015) - film
    0.667  The Tai Chi Master (2022) - film
 bm25:
    2.274  Monk Comes Down the Mountain (2015) - film
    2.047  The 18 Bronzemen (1976) - film
    1.832  The Miracle of Marcelino (1955) - film

=== eerie arthouse film about an alien seductress
 vector:
    0.695  The Abyss (1989) - film
    0.684  Hell Hole (2024) - film
    0.683  Alien Invasion: S.U.M.1 (2017) -

## 4. Hybrid search with Reciprocal Rank Fusion 🤝

How do you merge a cosine similarity (0 to 1) with a BM25 score (0 to 30+)?
You don't merge **scores**, you merge **ranks**:

$$\text{RRF}(d) = \sum_{\text{retrievers}} \frac{1}{k + \text{rank}(d)} \qquad k = 60$$

A film ranked 1st by one retriever and 3rd by the other beats a film ranked 1st by only one.

In [11]:
# By hand, on a toy example
k = 60
vector_ranking = ["Life of Pi", "The Abyss", "Noah"]
bm25_ranking   = ["The Abyss", "Life of Pi", "Fortress"]

scores = {}
for ranking in (vector_ranking, bm25_ranking):
    for rank, title in enumerate(ranking, start=1):
        scores[title] = scores.get(title, 0) + 1 / (k + rank)

for title, s in sorted(scores.items(), key=lambda x: -x[1]):
    print(f"{s:.4f}  {title}")

0.0325  Life of Pi
0.0325  The Abyss
0.0159  Noah
0.0159  Fortress


In [12]:
for q in queries:
    print(f"\n=== {q}")
    show(index.hybrid_search(q, top_k=3), 3)


=== Paganini
    0.033  The Devil's Violinist (2013) - film
    0.016  Black Narcissus (1947) - film
    0.016  The Last Witch Hunter (2015) - film

=== a man stranded on a lifeboat with a wild animal
    0.033  Life of Pi (2012) - film
    0.030  The Protector (2005) - film
    0.029  Noah (2014) - film

=== Shaolin monks
    0.033  The 18 Bronzemen (1976) - film
    0.033  Monk Comes Down the Mountain (2015) - film
    0.030  The 8 Diagram Pole Fighter (1984) - film

=== eerie arthouse film about an alien seductress
    0.032  A Quiet Place Part II (2021) - film
    0.032  Alien Invasion: S.U.M.1 (2017) - film
    0.030  Under the Skin (2014) - film


## 5. The reranker 🎯

Embeddings are a **bi-encoder**: query and documents are embedded *separately*, so the
document vectors can be pre-computed. Fast, but the model never sees query and document together.

A **cross-encoder** reads `(query, document)` *together* in one Transformer pass and outputs a
relevance score. Much sharper, much slower: so we only use it on the top ~20 candidates.

**retrieve wide and cheap → rerank narrow and expensive**

In [13]:
from movierag.models import load_reranker
reranker = load_reranker(cfg)

q = "a slow, unsettling science fiction film with almost no dialogue"
print("hybrid:");          show(index.search(q, "hybrid", top_k=5))
print("hybrid + rerank:"); show(index.search(q, "hybrid", top_k=5, reranker=reranker))

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1671.94it/s]


hybrid:
    0.032  Devs (2020) - series
    0.032  Under the Skin (2014) - film
    0.031  Automata (2014) - film
    0.030  Alien Invasion: S.U.M.1 (2017) - film
    0.030  Calculator (2014) - film
hybrid + rerank:
   -1.646  Under the Skin (2014) - film
   -3.949  A Quiet Place Part II (2021) - film
   -5.092  The Abyss (1989) - film
   -5.517  Hell Hole (2024) - film
   -5.925  Vivarium (2019) - film


## 6. Filters: the part embeddings can't do 🚧

"Under 90 minutes" is not a vibe, it's a number. Vector similarity will happily return a
3-hour epic that *talks about* being short.

So hard constraints are applied as **metadata filters**: we compute the allowed IDs in Python,
and Chroma only searches among those (`item_id IN [...]`).

In [15]:
allowed = {t.item_id for t in titles if t.kind == "series"}
print(len(allowed), "series in your list\n")
show(index.search("war and soldiers", "hybrid", top_k=5, allowed_ids=allowed))

19 series in your list

    0.032  See (2019) - series
    0.031  Raised by Wolves (2020) - series
    0.031  Tiger and Crane (2023) - series
    0.030  Avatar: The Last Airbender (2024) - series
    0.016  Doctor Who (2005) - series


# ✅ What we just built

| Retriever | Good at | Bad at |
|---|---|---|
| Vector | paraphrases, mood, "something like..." | exact names, rare words, numbers |
| BM25 | names, rare words | synonyms, paraphrase |
| Hybrid (RRF) | both | needs two indexes |
| + Reranker | precision at the top | slow, so only on a shortlist |
| Filters | hard constraints | nothing fuzzy |

We'll **measure** which combination works best in notebook 04, instead of trusting this table.

## Next: put an LLM on top 🐉